In [20]:
# Montar Google Drive (si ejecutamos en Google Colab) y definir directorio de datos
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data_dir = '/content/drive/MyDrive/NLP/Practica/'
except ImportError:
    data_dir = './'

# Nos aseguramos de que el directorio exista
os.makedirs(data_dir, exist_ok=True)

# Cargamos el dataset y variables del notebook anterior
import pickle
with open(os.path.join(data_dir, 'df_eda.pkl'), 'rb') as f:
    df, median_words_rw = pickle.load(f)

# Definimos la función bold para mostrar texto en negrita
def bold(text):
    return f"\033[1m{text}\033[0m"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2 - Preprocesamiento

In [21]:
# Dependencias
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
import re
import numpy as np
import pandas as pd
import random
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [22]:
# NLP Preprocessing pipeline
class nlp_pipeline:
  '''
  nlp_pipeline realiza el preprocesado del corpus
  antes del entrenamiento.
  Parámetros:
  -----------
  -> Array de strings
  <- Array de string preprocesados

  Tratamiento:
  ------------
  - minúsculas
  - eliminar caracteres
  - tokenizar
  - quitar stop words
  - pad sequence (tamaño medio reviews)
  '''
  def __init__(self, corpus):
    self.corpus = corpus
    self.stop_words_list = set(stopwords.words('english'))

  def preprocessing(self):
      prep_corpus = self.corpus.copy()
      for pos, line in enumerate(prep_corpus):
        line = self.to_lower(line)
        line = self.leave_nonword_chars(line)
        prep_corpus[pos] = self.leave_stop_words(line)

      return np.array(self.pad_sequence(prep_corpus))

  def to_lower(self, param:str):
    '''
    Get a parameter string
    Return the same content but in lowercase
    '''

    return param.lower()

  def leave_nonword_chars(self, param:str):
    '''
    Get a parameter string
    Return the non alphanumeric characters
    '''
    return re.sub('[^a-zA-Z0-9 ]', '', param)

  def leave_stop_words(self, param:str):
    '''
    Get a parameter string
    Return the non stopword words
    '''
    return " ".join([word for word in param.split() if word not in self.stop_words_list])

  def pad_sequence(self, param: list):
    '''
    Get a parameter list
    Return a new list with pad sequence as a result
    of the mean length of the list text lines
    '''

    # Give a list with the size of each line
    list_of_words = [len(line.split())  for line in param]

    # Calculate the median number of words in the list of lines
    median_words = np.median(list_of_words)

    int_median_words = int(np.round(median_words, 0))

    # Truncate all the lines to the median length of words
    trunc_list = []
    for line in param:
      temp_low=line.split() #temp list of words by line

      trunc_list.append(" ".join(temp_low[:int_median_words]))

    # Returns truncated list with the default pad_sequence
    return trunc_list



In [23]:
#pd.set_option('display.max_colwidth', None)
df[['reviewText', 'overall']][:5]

,reviewText,overall
0,"Not much to write about here, but it does exac...",5.0
1,The product does exactly as it should and is q...,5.0
2,The primary job of this device is to block the...,5.0
3,Nice windscreen protects my MXL mic and preven...,5.0
4,This pop filter is great. It looks and perform...,5.0


In [24]:
# Añadimos una columna para la clasificación binaria
'''
Criterio binario para análisis de sentimiento
Overall: 0-5
Distribución:
Overall     Sentimiento (1-positivo, 0-negativo)
-------     -----------
0-3           0
4-5           1
'''

df['bin_label'] = df.apply(lambda row: 1 if row['overall'] >= 4 else 0, axis=1)

In [25]:
nlp_pipe_inst = nlp_pipeline(df['reviewText'])
preproc_corpus = nlp_pipe_inst.preprocessing()
preproc_corpus[:5]

array(['much write exactly supposed filters pop sounds recordings much crisp one lowest prices pop filters amazon might well buy honestly work despite pricing',
       'product exactly quite affordablei realized double screened arrived even better expectedas added bonus one screens carries small hint smell old grape candy used buy reminiscents sake cannot',
       'primary job device block breath would otherwise produce popping sound allowing voice pass noticeable reduction volume high frequencies double cloth filter blocks pops lets voice coloration metal',
       'nice windscreen protects mxl mic prevents pops thing gooseneck marginally able hold screen position requires careful positioning clamp avoid sagging',
       'pop filter great looks performs like studio filter youre recording vocals eliminate pops gets recorded sing'],
      dtype='<U372')

In [26]:
# save dataset


In [27]:
# BoW - CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(analyzer='word', ngram_range=(1, 1))
X2 = vectorizer.fit_transform(preproc_corpus)
X2.sum(axis=0)
total_counts = X2.sum(axis=0).tolist()[0]
words = vectorizer.get_feature_names_out()

In [28]:
# Veces que se repite una palabra (muestra)
word_counts = dict(zip(words, total_counts))
list(word_counts.items())[7250:7280]

[('guarded', 1),
 ('guardian', 3),
 ('guards', 1),
 ('guess', 84),
 ('guessed', 2),
 ('guessing', 7),
 ('guessnot', 1),
 ('guesswork', 1),
 ('guest', 1),
 ('guiatars', 1),
 ('guidance', 1),
 ('guide', 12),
 ('guidelines', 1),
 ('guides', 13),
 ('guidescovers', 1),
 ('guidetree', 1),
 ('guiitar', 2),
 ('guild', 6),
 ('guilty', 2),
 ('guitalele', 1),
 ('guitar', 3184),
 ('guitaramplifieretc', 1),
 ('guitarand', 2),
 ('guitaraudio', 1),
 ('guitarbanjo', 1),
 ('guitarbanjomandolin', 1),
 ('guitarbases', 1),
 ('guitarbass', 4),
 ('guitarbassits', 1),
 ('guitarbrand', 1)]

## Features

In [29]:
from sklearn.model_selection import train_test_split

In [30]:
# creo mi columna de referencia
ground_truth = df['bin_label']


In [31]:
x_train, x_test, y_train, y_test = train_test_split(
    preproc_corpus,
    ground_truth,
    train_size=0.75,
    test_size=0.25,
    random_state=42,
    shuffle=True,
    stratify=ground_truth
)

In [32]:
# Extracción características TF-IDF
tfidf = TfidfVectorizer(
    max_df=0.95,
    min_df=5,
    max_features=2500,
    strip_accents='ascii',
    ngram_range=(1, 1)
)
tfidf.fit(x_train)

TfidfVectorizer(max_df=0.95, max_features=2500, min_df=5, strip_accents='ascii')

In [33]:
print(list(tfidf.vocabulary_.items())[:20])

[('good', np.int64(904)), ('dont', np.int64(615)), ('see', np.int64(1864)), ('better', np.int64(232)), ('basic', np.int64(201)), ('strings', np.int64(2096)), ('price', np.int64(1646)), ('range', np.int64(1711)), ('work', np.int64(2456)), ('well', np.int64(2416)), ('length', np.int64(1172)), ('somewhat', np.int64(2003)), ('appears', np.int64(142)), ('level', np.int64(1180)), ('hosa', np.int64(1016)), ('cable', np.int64(307)), ('quality', np.int64(1700)), ('place', np.int64(1571)), ('products', np.int64(1663)), ('sound', np.int64(2014))]


In [34]:
x_train_ = tfidf.transform(x_train)
x_test_ = tfidf.transform(x_test)

In [35]:
# Some review words (TF-IDF)
i = random.randint(0, len(x_train))


doc_vector = x_train_[i]
df_tfidf = pd.DataFrame(doc_vector.T.todense(), index=tfidf.get_feature_names_out(), columns=['tfidf'])
df_tfidf = df_tfidf[df_tfidf['tfidf'] > 0]

top_n = 10


In [36]:
print(bold('ID        :')+' {}'.format(i))
print(bold('Sentiment :')+' {}'.format(y_train.iloc[i]))
print(bold('Review    :')+' {} \n'.format(x_train[i]))

print('Top {} words with highest TF_IDF in the review {}:\n{}'.format(top_n, i, df_tfidf.sort_values(by=["tfidf"],ascending=False)[:top_n]))
print('\nTop {} words with lowest TF_IDF in the review {}:\n{}'.format(top_n, i, df_tfidf.sort_values(by=["tfidf"],ascending=False)[-top_n:]))

ID        : 1842
Sentiment : 0
Review    : wish bought longer cables cables fine supposed know spend dollar dollar bills get higher quality cable tonal difference wouldnt anything significant 

Top 10 words with highest TF_IDF in the review 1842:
               tfidf
dollar      0.514825
cables      0.353616
tonal       0.276726
higher      0.240382
spend       0.233984
wouldnt     0.226786
supposed    0.226240
wish        0.216652
difference  0.213378
anything    0.205874

Top 10 words with lowest TF_IDF in the review 1842:
               tfidf
wish        0.216652
difference  0.213378
anything    0.205874
longer      0.203348
know        0.180095
fine        0.173045
cable       0.165475
quality     0.134056
bought      0.131837
get         0.129882


In [37]:
# Guardamos los datos preprocesados para los siguientes notebooks
import pickle
with open(os.path.join(data_dir, 'preprocessed_data.pkl'), 'wb') as f:
    pickle.dump((x_train, x_test, y_train, y_test, tfidf, x_train_, x_test_, median_words_rw), f)
